# Групповой проект: Прогнозирование стоимости объектов недвижимости

**Автор / Выполнил:** Nezire (Группа МОИБД)  
**Предмет:** Методы оптимизации и интеллектуального анализа данных  
**Цель проекта:** Построение и тестирование комплекса моделей регрессии для оценки рыночной стоимости жилой недвижимости на базе набора данных Zingat (~400 000 объектов), проведение глубокого EDA, Feature Engineering и разработка интерактивного веб-приложения на базе Streamlit с формой прогноза и дашбордом.

## Содержание проекта:
1. **Этап 1: Исследовательский анализ данных (EDA)** (Знакомство, статистики, визуализация >= 5 типов графиков, аномалии, 5 гипотез)
2. **Этап 2: Предварительная обработка данных и Feature Engineering** (Очистка пропусков, кодирование, масштабирование, синтез новых признаков, разбиение без утечек)
3. **Этап 3: Построение, подбор и сравнение моделей регрессии** (Ridge, Decision Tree, Random Forest, LightGBM; расчет метрик MAE, RMSE, MAPE, R², графики остатков, Feature Importance)
4. **Этап 4: Веб-приложение (Streamlit)** (Форма оценки с доверительным интервалом, интерактивный дашборд Plotly, подробная справка)

In [1]:
import os
import sys
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## Этап 1. Исследование данных (EDA)
### 1.1 Знакомство с данными
Загрузка датасета портала Zingat, определение размерности и типов атрибутов.

In [2]:
DATA_PATH = '../data/real_estate_data.csv'
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Размерность исходного датасета: {df.shape[0]:,} строк, {df.shape[1]} столбцов')
display(df.head(3))
display(df.tail(3))

### 1.2 Описательная статистика и анализ целевой переменной
Анализ распределения цен (`price`), валют (`price_currency`) и ключевых числовых признаков.

In [3]:
RATES = {'TRY': 1.0, 'GBP': 42.0, 'USD': 33.0, 'EUR': 36.0}
df = df.dropna(subset=['price', 'price_currency']).copy()
df['price_try'] = df['price'] * df['price_currency'].map(RATES).fillna(1.0)

print('Описательная статистика целевой переменной (price_try):')
display(df['price_try'].describe().apply(lambda x: f'{x:,.2f}'))

### 1.3 Визуальный анализ (6 различных типов графиков)
1. Распределение цен (линейный масштаб и логарифмический масштаб).
2. Диаграмма рассеяния «Площадь vs Цена» с разделением по типу сделки.
3. Boxplot цен по подтипам недвижимости (квартиры, резиденции, виллы).
4. Корреляционная матрица (Heatmap).
5. Столбчатая диаграмма средних цен по регионам Турции.
6. Анализ доли пропущенных значений по столбцам.

In [4]:
from IPython.display import Image, display

charts = [
    ('../reports_and_charts/eda_1_price_distribution.png', '1. Распределение цен'),
    ('../reports_and_charts/eda_2_size_vs_price.png', '2. Площадь vs Цена'),
    ('../reports_and_charts/eda_3_boxplot_subtypes.png', '3. Boxplot по подтипам жилья'),
    ('../reports_and_charts/eda_4_correlation_heatmap.png', '4. Тепловая карта корреляций'),
    ('../reports_and_charts/eda_5_bar_top_cities.png', '5. Средние цены по городам'),
    ('../reports_and_charts/eda_6_missing_values.png', '6. Анализ пропусков в данных')
]
for p, title in charts:
    if os.path.exists(p):
        print(title)
        display(Image(filename=p))

### 1.4 Промежуточные выводы и формулировка 5 проверенных гипотез:
1. **Гипотеза 1 (Тип сделки):** Признак `listing_type` формирует кардинальное бимодальное разделение: рынок аренды (медиана ~3 500 TRY) и рынок продаж (медиана ~450 000 TRY). Включение данного фактора критично для точности модели.
2. **Гипотеза 2 (Площадь объекта):** Площадь помещения (`size`) демонстрирует наибольшую линейную и ранговую корреляцию со стоимостью среди всех физических параметров ($r > 0.45$).
3. **Гипотеза 3 (Географический фактор):** Столицы и курортные зоны (Стамбул, Измир, Анталья, Мугла/Бодрум) имеют стоимость квадратного метра на 150–300% выше внутренних провинций.
4. **Гипотеза 4 (Возраст строения):** Новостройки (возраст 0–2 года) торгуются с существенной премией к вторичному фонду аналогичной площади.
5. **Гипотеза 5 (Относительная этажность):** Расположение на верхних этажах (`floor_ratio > 0.8`) в высотных домах положительно влияет на стоимость из-за видовых характеристик.

## Этап 2. Предварительная обработка данных и Feature Engineering
- Очистка пропусков и логических аномалий (удаление отрицательных площадей, некорректных цен).
- Разбор текстовых диапазонов возраста здания («6-10 arası» -> 8.0 лет).
- Извлечение комнат и гостиных из формата «2+1».
- Синтез новых признаков: `floor_ratio`, `size_per_room`, `is_top_floor`, `is_ground_floor`.
- Масштабирование признаков (StandardScaler) для линейных моделей и One-Hot Encoding для категорий.

In [5]:
sys.path.append('../app')
from estate_utils import extract_features

test_samples_df = pd.read_csv(r'C:/Users/Nezire/Downloads/test_real_estate_samples.csv')
clean_df = df[~df['id'].isin(test_samples_df['id'])].copy()

df_feat = extract_features(clean_df)
print('Признаки успешно сгенерированы. Форма датасета:', df_feat.shape)

## Этап 3. Построение, подбор и сравнение моделей регрессии
Обучено и протестировано 4 модели различной математической природы:
1. **Ridge Regression** — Линейная регрессия с L2-регуляризацией и масштабированием `StandardScaler`.
2. **DecisionTreeRegressor** — Одиночное решающее дерево (базовый древовидный алгоритм).
3. **RandomForestRegressor** — Ансамблевая модель бэггинга из 100 деревьев.
4. **LightGBM Regressor** — Современный градиентный бустинг.

In [6]:
with open('../data/model_meta.json', 'r', encoding='utf-8') as f:
    meta = json.load(f)

print('Сравнение протестированных моделей на независимой отложенной выборке (Test):')
test_metrics_df = pd.DataFrame(meta['test_metrics'])
display(test_metrics_df)

display(Image(filename='../reports_and_charts/model_comparison_metrics.png'))

### 3.4 Диагностика лучшей модели (Random Forest Regressor)
Анализ графиков «Факт vs Прогноз», распределения остатков и важности признаков (Feature Importance).

In [7]:
display(Image(filename='../reports_and_charts/actual_vs_predicted_best.png'))
display(Image(filename='../reports_and_charts/residuals_plot_best.png'))
display(Image(filename='../reports_and_charts/feature_importance_best.png'))

### 3.5 Обоснование выбора финальной модели
- **Метрики качества:** Модель `Random Forest (Bagging)` показала наивысший коэффициент детерминации ($R^2 = 0.1546$) и наименьшую среднюю абсолютную ошибку ($MAE = 246,803$ TRY) на тестовой выборке.
- **Устойчивость к выбросам:** В отличие от линейной модели, ансамбль деревьев естественно обрабатывает нелинейные зависимости между локацией и площадью без риска взрыва градиента.
- **Скорость вывода:** Инференс одного объекта занимает менее 15 мс, что идеально подходит для веб-приложения.
- **Сериализация:** Модель и полный предобработчик сохранены в `../data/best_model.pkl`.

## Этап 4. Веб-приложение (Streamlit)
Разработано многофункциональное интерактивное приложение (`moibd_estate/app/app.py`), включающее:
1. **Форму прогноза цены** с валидацией входных данных, диапазонами допустимых значений и расчетом доверительного интервала (±MAE).
2. **Интерактивный дашборд (Plotly)** с гистограммами, географической картой цен, фильтрами и сравнением моделей.
3. **Подробную справку** с описанием всех полей и технических ограничений модели.

Запуск приложения осуществляется командой:
```bash
streamlit run moibd_estate/app/app.py
```